In [ ]:
from google.colab import auth
auth.authenticate_user()

import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
DATASET_ID = "produccion"
TABLE_ID= "TRAMAS_diarias_BBVA_2026"
NOMBRE_TRAMA= 'RED101126_1111.TXT'

In [ ]:
bigquery_client = bigquery.Client(project=PROJECT_ID)
def Ejecutar_Query(trama):
    tabla_tramas = bigquery_client.dataset(DATASET_ID).table(TABLE_ID)
    print(f"⏳ Limpiando Tramas del archivo '{NOMBRE_TRAMA}' ....")
    query = f"""
        SELECT TIPO_DE_SEGURO, TIPO_DE_MOVIMIENTO,
              REGEXP_REPLACE(TRAMA_ORIGINAL, '[^\\x20-\\x7E\\n\\r]|\\t', ' ') AS TRAMA_LIMPIA
        FROM `{tabla_tramas}`
        WHERE NOMBRE_DE_ARCHIVO = @archivo_txt
        ORDER BY TIPO_DE_SEGURO
        """
    job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ScalarQueryParameter("archivo_txt", "STRING", trama)])
    df = bigquery_client.query(query, job_config=job_config).to_dataframe()
    return df

df_trama= Ejecutar_Query(NOMBRE_TRAMA)
archivo_salida= f"/content/{NOMBRE_TRAMA.split(".")[0]} - limpia.TXT"

with open(archivo_salida, "w", encoding="utf-8") as f:
    for linea in df_trama["TRAMA_LIMPIA"]:
        f.write(str(linea)+ "\n")

print(f"✅ Archivo generado: {archivo_salida}")

⏳ Limpiando Tramas del archivo 'RED101126_1111.TXT' ....
✅ Archivo generado: /content/RED101126_1111 - limpia.TXT
